# Assignment 1 — Custom Missing-Value Imputer

**Course:** Feature Engineering & MLOps
**Topic:** Custom Imputer Class (Missing Value Handling)
**Dataset:** `student_performance_raw.csv` (PrepEdge coaching institute data, Unit 1)

In this notebook I build my own `CustomImputer` class that follows the same `fit` / `transform`
pattern as scikit-learn's built-in imputers, apply it to the dataset, verify it works, and answer
the reflection questions.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted
import pandas.api.types as ptypes

pd.set_option('display.max_columns', None)

## 2. Load the dataset and look at missing values

First I load the raw CSV and check which columns actually have missing values, and how many,
so I know what my imputer needs to handle.

In [2]:
df = pd.read_csv('student_performance_raw.csv')
print(df.shape)
df.head()

(600, 17)


,student_id,city,city_tier,course,batch_type,age,enrollment_date,attendance_pct,weekly_study_hours,income_bracket,prev_exam_score,mock_test_1,mock_test_2,mock_test_3,doubt_sessions_attended,feedback_text,final_score
0,1447,Patna,2,JEE Main,Weekday,18,2026-05-27,91.1,0.8,<5L,65.2,68.1,59.0,72.2,1,Need more practice sheets for weak topics,42.0
1,1405,Patna,2,NEET,Weekday,16,2024-07-08,91.9,2.4,NaN,64.9,82.0,88.4,100.0,3,Would like more one-on-one mentoring,62.0
2,1510,Mumbai,1,JEE Main,Weekday,18,2025-03-17,67.1,6.7,5-10L,90.1,86.5,91.1,76.4,5,"Great teaching pace, doubts cleared quickly",56.4
3,1456,Delhi,1,NEET,Weekday,18,2025-06-16,70.7,2.0,10-20L,29.7,19.4,24.9,NaN,3,Need more practice sheets for weak topics,31.7
4,1202,Patna,2,JEE Advanced,Weekday,17,2026-04-16,69.3,6.5,>20L,70.0,63.8,74.5,69.1,4,Need more practice sheets for weak topics,44.5


In [3]:
# how many missing values per column?
df.isna().sum().sort_values(ascending=False)

feedback_text              68
weekly_study_hours         36
income_bracket             30
prev_exam_score            25
mock_test_3                21
student_id                  0
city                        0
course                      0
city_tier                   0
attendance_pct              0
enrollment_date             0
age                         0
batch_type                  0
mock_test_2                 0
mock_test_1                 0
doubt_sessions_attended     0
final_score                 0
dtype: int64

This matches what the assignment sheet says: `weekly_study_hours`, `prev_exam_score`,
`mock_test_3`, `income_bracket` and `feedback_text` have missing values, each for a different
reason (MCAR / MAR / MNAR / missing categorical / missing text).

## 3. Train / test split

I split the data 80/20 **before** doing anything else. This is important because the imputer's
fill values (mean, median, mode) must only ever be learned from the training data — otherwise
information from the test set would "leak" into training, which defeats the purpose of having a
test set in the first place.

In [4]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (480, 17)
Test shape : (120, 17)


## 4. The `CustomImputer` class

I built this class from scratch, following the same pattern as `sklearn.impute.SimpleImputer`:

- `fit(X, y=None)` learns the fill value for every column (mean/median for numeric columns, the
  mode for categorical/text columns) and stores them in `self.fill_values_`.
- `transform(X)` uses the values that were already learned in `fit` (it never recomputes
  anything) and returns a filled **copy** of the data. If `add_missing_indicator=True`, it also
  adds a `<column>_was_missing` flag column for every column that had missing values during
  training.
- Column type (numeric vs categorical) is detected automatically using `pandas.api.types`, so I
  don't have to tell it manually which columns are which.
- If someone calls `transform()` before `fit()`, it raises a clear error using sklearn's
  `check_is_fitted`.
- **Bonus:** a `column_overrides` parameter lets individual columns use a different strategy than
  the dataset-wide default (see section 8).

In [5]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """
    A simple missing-value imputer that follows scikit-learn's fit/transform API.

    For every column, a single fill value is learned during `fit` (mean or median for
    numeric columns, the most frequent value for categorical/text columns) and that same
    value is reused every time `transform` is called - it is never recomputed on new data.
    This is what keeps the imputer leakage-safe: fit it once on the training set, then just
    reuse it on the test set (or any future data).

    Parameters
    ----------
    numeric_strategy : str, default='median'
        Strategy used for numeric columns. One of {'mean', 'median'}.
    categorical_strategy : str, default='most_frequent'
        Strategy used for categorical / text columns. Currently only 'most_frequent'
        (the mode) is supported.
    add_missing_indicator : bool, default=True
        If True, an extra binary column '<col>_was_missing' is added for every column
        that had at least one missing value in the training data.
    column_overrides : dict, default=None
        Optional dict of {column_name: strategy} to override the default strategy for
        specific columns, e.g. {'weekly_study_hours': 'mean'}. (Bonus feature.)

    Attributes
    ----------
    fill_values_ : dict
        The fill value learned for each column. Only set after `fit` is called.
    missing_cols_ : list
        Columns that had at least one missing value in the training data. Used to decide
        which '_was_missing' indicator columns to create.
    """

    def __init__(self, numeric_strategy='median', categorical_strategy='most_frequent',
                 add_missing_indicator=True, column_overrides=None):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _is_numeric(self, series):
        """Return True if a pandas Series should be treated as a numeric column."""
        return ptypes.is_numeric_dtype(series)

    def _strategy_for(self, col, is_numeric):
        """Work out which strategy to use for a given column (checks overrides first)."""
        overrides = self.column_overrides or {}
        if col in overrides:
            return overrides[col]
        return self.numeric_strategy if is_numeric else self.categorical_strategy

    def fit(self, X, y=None):
        """
        Learn the fill value for every column in X.

        Parameters
        ----------
        X : pandas.DataFrame
            The training data. Should NOT include the test set (see reflection Q1).
        y : ignored
            Not used, present for API consistency with scikit-learn.

        Returns
        -------
        self
        """
        self.fill_values_ = {}
        self.missing_cols_ = []

        for col in X.columns:
            series = X[col]
            is_numeric = self._is_numeric(series)
            strategy = self._strategy_for(col, is_numeric)

            if strategy == 'mean':
                fill_value = series.mean()
            elif strategy == 'median':
                fill_value = series.median()
            elif strategy == 'most_frequent':
                mode = series.mode(dropna=True)
                fill_value = mode.iloc[0] if not mode.empty else np.nan
            else:
                raise ValueError(f"Unknown strategy '{strategy}' for column '{col}'")

            self.fill_values_[col] = fill_value

            if series.isna().any():
                self.missing_cols_.append(col)

        return self

    def transform(self, X):
        """
        Fill missing values in X using the values learned during `fit`.

        Parameters
        ----------
        X : pandas.DataFrame
            Data to transform (can be the training data, the test data, or new data).
            Must have the same columns that were seen during `fit`.

        Returns
        -------
        pandas.DataFrame
            A COPY of X with missing values filled in, plus '_was_missing' indicator
            columns if add_missing_indicator=True.
        """
        check_is_fitted(self, 'fill_values_')
        X = X.copy()

        # add the "was this missing?" flags first, using the ORIGINAL missingness pattern
        if self.add_missing_indicator:
            for col in self.missing_cols_:
                if col in X.columns:
                    X[f'{col}_was_missing'] = X[col].isna().astype(int)

        # now fill in the actual values, using ONLY the values stored during fit
        for col, fill_value in self.fill_values_.items():
            if col in X.columns:
                X[col] = X[col].fillna(fill_value)

        return X

In [6]:
# quick smoke test: transform() should refuse to run before fit()
try:
    CustomImputer().transform(train_df)
except Exception as e:
    print(f"Got the expected error: {type(e).__name__}: {e}")

Got the expected error: NotFittedError: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## 5. Apply the imputer to the dataset

I only pass the columns that actually need imputing (the numeric ones plus `income_bracket`
and `feedback_text`) rather than the whole dataframe, since columns like `student_id` or
`enrollment_date` don't need this kind of imputation.

In [7]:
cols_to_impute = ['weekly_study_hours', 'prev_exam_score', 'mock_test_3',
                   'income_bracket', 'feedback_text']

imputer = CustomImputer(numeric_strategy='median', categorical_strategy='most_frequent',
                         add_missing_indicator=True)

# fit on TRAIN ONLY
imputer.fit(train_df[cols_to_impute])

print("Learned fill values:")
imputer.fill_values_

Learned fill values:


{'weekly_study_hours': np.float64(5.0),
 'prev_exam_score': np.float64(66.1),
 'mock_test_3': np.float64(71.3),
 'income_bracket': '5-10L',
 'feedback_text': 'Need more practice sheets for weak topics'}

In [8]:
# use the SAME fitted imputer to transform both splits
train_imputed = imputer.transform(train_df[cols_to_impute])
test_imputed = imputer.transform(test_df[cols_to_impute])

train_imputed.head()

,weekly_study_hours,prev_exam_score,mock_test_3,income_bracket,feedback_text,weekly_study_hours_was_missing,prev_exam_score_was_missing,mock_test_3_was_missing,income_bracket_was_missing,feedback_text_was_missing
145,2.0,56.5,54.5,5-10L,Would like more one-on-one mentoring,0,0,0,0,0
9,5.0,57.8,34.9,10-20L,Need more practice sheets for weak topics,1,0,0,0,0
375,6.2,45.1,65.9,5-10L,Need more practice sheets for weak topics,0,0,0,0,0
523,1.1,54.3,74.0,5-10L,Mock tests really helped identify gaps,0,0,0,0,0
188,2.8,78.8,87.9,<5L,"Great teaching pace, doubts cleared quickly",0,0,0,0,0


## 6. Verify there are no missing values left

In [9]:
assert train_imputed[cols_to_impute].isna().sum().sum() == 0, "Missing values remain in train!"
assert test_imputed[cols_to_impute].isna().sum().sum() == 0, "Missing values remain in test!"

print("Train missing values after imputation:", train_imputed[cols_to_impute].isna().sum().sum())
print("Test missing values after imputation :", test_imputed[cols_to_impute].isna().sum().sum())
print("\nAll good, no missing values left in either split.")

Train missing values after imputation: 0
Test missing values after imputation : 0

All good, no missing values left in either split.


In [10]:
# and the new indicator columns are there too
[c for c in train_imputed.columns if c.endswith('_was_missing')]

['weekly_study_hours_was_missing',
 'prev_exam_score_was_missing',
 'mock_test_3_was_missing',
 'income_bracket_was_missing',
 'feedback_text_was_missing']

## 7. Mean and standard deviation before vs. after imputation

For the numeric columns, I compare the mean and std of the **training split** before and after
imputation.

In [11]:
numeric_cols = ['weekly_study_hours', 'prev_exam_score', 'mock_test_3']

before_after = pd.DataFrame({
    'mean_before': train_df[numeric_cols].mean(),
    'mean_after': train_imputed[numeric_cols].mean(),
    'std_before': train_df[numeric_cols].std(),
    'std_after': train_imputed[numeric_cols].std(),
})
before_after

,mean_before,mean_after,std_before,std_after
weekly_study_hours,6.064176,6.008750,4.350251,4.241815
prev_exam_score,65.824508,65.837708,14.370793,14.021654
mock_test_3,70.701505,70.720208,19.077301,18.776510


**Comment:** The mean barely moves for any of the three columns since median imputation
fills in a "typical" value close to the middle of the distribution. The standard deviation
drops a little after imputation in every column - this makes sense, because we're replacing
missing values with a single fixed number instead of real (varying) data, which slightly
shrinks the spread of the column. The effect is small here because only a small fraction of
each column was missing.

## 8. Sanity check against scikit-learn's `SimpleImputer`

If my class is correct, it should produce the exact same fill values as `SimpleImputer` when
using an equivalent strategy on the same numeric columns.

In [12]:
sklearn_imputer = SimpleImputer(strategy='median')
sklearn_imputer.fit(train_df[numeric_cols])

my_values = [imputer.fill_values_[c] for c in numeric_cols]
sklearn_values = list(sklearn_imputer.statistics_)

comparison = pd.DataFrame({
    'column': numeric_cols,
    'my_fill_value': my_values,
    'sklearn_fill_value': sklearn_values,
})
comparison['match'] = np.isclose(comparison['my_fill_value'], comparison['sklearn_fill_value'])
comparison

,column,my_fill_value,sklearn_fill_value,match
0,weekly_study_hours,5.0,5.0,True
1,prev_exam_score,66.1,66.1,True
2,mock_test_3,71.3,71.3,True


In [13]:
assert comparison['match'].all(), "Fill values don't match SimpleImputer!"
print("All fill values match SimpleImputer's own median calculation. Core logic is correct.")

All fill values match SimpleImputer's own median calculation. Core logic is correct.


## 9. Bonus — per-column strategy override

Here I use the `column_overrides` parameter to make `weekly_study_hours` use the **mean**
instead of the dataset-wide default (`median`), while `income_bracket` explicitly stays on
`most_frequent`. This shows two different columns using two different strategies in the same
fit call.

In [14]:
bonus_imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    add_missing_indicator=True,
    column_overrides={
        'weekly_study_hours': 'mean',
        'income_bracket': 'most_frequent',
    }
)

bonus_imputer.fit(train_df[cols_to_impute])

print("weekly_study_hours -> mean override  :", bonus_imputer.fill_values_['weekly_study_hours'])
print("(compare) plain median for that col  :", train_df['weekly_study_hours'].median())
print("income_bracket -> most_frequent      :", bonus_imputer.fill_values_['income_bracket'])

weekly_study_hours -> mean override  : 6.064175824175824
(compare) plain median for that col  : 5.0
income_bracket -> most_frequent      : 5-10L


The mean and median for `weekly_study_hours` are close but not identical, so the override
is clearly doing something different from the default median strategy - confirming the
per-column override works as intended.

## 10. Reflection Questions

**1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?**

If I fit on the full dataset (or on the test set), the fill values would be calculated using
information from the test rows too. That means the test set is no longer truly "unseen" data -
it has quietly influenced the numbers used to fill it. This is a form of data leakage, and it
makes any model evaluation on that test set overly optimistic and unreliable, because in the
real world we won't have future/test data available at training time.

**2. `mock_test_3` is MNAR (Session 4). Does mean/median imputation genuinely solve the problem for this column? What does `add_missing_indicator` contribute that plain imputation does not?**

No, not really. `mock_test_3` is missing *because* its value is low (students who scored badly
are more likely to not have a value recorded), so filling those spots with the median actually
pulls those students' scores upward artificially and hides the pattern instead of fixing it.
Plain imputation only solves the "no NaNs left" problem, not the underlying bias. The
`_was_missing` indicator column at least lets a downstream model separately learn that "this
value was originally missing" as its own signal, so the model has a chance to account for the
fact that these rows are systematically different, rather than treating the imputed value as if
it were a real observed score.

**3. Suppose a brand-new column, entirely missing in the training data but present in the test data, is passed to your imputer. What does your current implementation do — and what SHOULD a production-grade version do instead?**

If a column is entirely missing (all NaN) in the training data, `.mean()`/`.median()`/`.mode()`
all return `NaN` (mode returns an empty Series in that case, which my code turns into `NaN`), so
`fill_values_` would store `NaN` for that column, and `transform()` would effectively fill
missing values with `NaN` - meaning the column doesn't actually get fixed at all. A
production-grade version should instead detect this case explicitly during `fit()` (e.g. check
if a column has zero valid values) and either raise a clear warning/error, or fall back to a
safe constant (like 0, or a placeholder category such as `'unknown'`) so it fails loudly and
predictably instead of silently leaving NaNs in the output.